# Installer CLI
> Command-line interface for MCP installation

In [ ]:
#| default_exp install.cli

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
import argparse
import sys
from typing import Optional, List

from nbdev_mcp.install import (
    PROVIDERS, install_all, uninstall_all, check_status, print_status,
    get_installer, MCPServerConfig, get_python_path
)

## CLI Implementation

In [ ]:
#| export
def create_parser() -> argparse.ArgumentParser:
    """Create argument parser for installer CLI."""
    parser = argparse.ArgumentParser(
        prog='nbdev-mcp-install',
        description='Install nbdev-mcp configuration for AI tools'
    )
    
    subparsers = parser.add_subparsers(dest='command', help='Command to run')
    
    # Install command
    install_parser = subparsers.add_parser('install', help='Install MCP configuration')
    install_parser.add_argument(
        'providers', nargs='*', default=None,
        help=f'Providers to install (default: all). Choices: {PROVIDERS}'
    )
    install_parser.add_argument(
        '--python', '-p', dest='python_path',
        help='Python interpreter path (default: current)'
    )
    install_parser.add_argument(
        '--force', '-f', action='store_true',
        help='Overwrite existing configuration'
    )
    install_parser.add_argument(
        '--dry-run', '-n', action='store_true',
        help='Show what would be done without making changes'
    )
    
    # Uninstall command
    uninstall_parser = subparsers.add_parser('uninstall', help='Uninstall MCP configuration')
    uninstall_parser.add_argument(
        'providers', nargs='*', default=None,
        help=f'Providers to uninstall (default: all). Choices: {PROVIDERS}'
    )
    uninstall_parser.add_argument(
        '--dry-run', '-n', action='store_true',
        help='Show what would be done without making changes'
    )
    
    # Status command
    status_parser = subparsers.add_parser('status', help='Check installation status')
    status_parser.add_argument(
        'providers', nargs='*', default=None,
        help=f'Providers to check (default: all). Choices: {PROVIDERS}'
    )
    status_parser.add_argument(
        '--json', '-j', action='store_true',
        help='Output as JSON'
    )
    
    return parser

In [ ]:
#| export
def validate_providers(providers: Optional[List[str]]) -> Optional[List[str]]:
    """Validate provider names.
    
    Returns None (meaning 'all') if empty list provided.
    Raises ValueError for invalid providers.
    """
    if not providers:
        return None
    
    invalid = [p for p in providers if p.lower() not in PROVIDERS]
    if invalid:
        raise ValueError(f"Unknown providers: {invalid}. Valid: {PROVIDERS}")
    
    return [p.lower() for p in providers]

In [ ]:
#| export
def cmd_install(args) -> int:
    """Handle install command."""
    try:
        providers = validate_providers(args.providers)
    except ValueError as e:
        print(f"Error: {e}", file=sys.stderr)
        return 1
    
    results = install_all(
        python_path=args.python_path,
        force=args.force,
        dry_run=args.dry_run,
        providers=providers
    )
    
    success = True
    for result in results:
        print(result.message)
        if not result.success:
            success = False
    
    return 0 if success else 1


def cmd_uninstall(args) -> int:
    """Handle uninstall command."""
    try:
        providers = validate_providers(args.providers)
    except ValueError as e:
        print(f"Error: {e}", file=sys.stderr)
        return 1
    
    results = uninstall_all(
        dry_run=args.dry_run,
        providers=providers
    )
    
    success = True
    for result in results:
        print(result.message)
        if not result.success:
            success = False
    
    return 0 if success else 1


def cmd_status(args) -> int:
    """Handle status command."""
    try:
        providers = validate_providers(args.providers)
    except ValueError as e:
        print(f"Error: {e}", file=sys.stderr)
        return 1
    
    if args.json:
        import json
        statuses = check_status(providers)
        print(json.dumps(statuses, indent=2))
    else:
        print_status(providers)
    
    return 0

In [ ]:
#| export
def main(args: Optional[List[str]] = None) -> int:
    """Main entry point for installer CLI.
    
    Parameters
    ----------
    args : list[str], optional
        Command-line arguments. Defaults to sys.argv[1:].
    
    Returns
    -------
    int
        Exit code (0 for success).
    """
    parser = create_parser()
    parsed = parser.parse_args(args)
    
    if parsed.command is None:
        # Default to status if no command
        print_status()
        return 0
    
    commands = {
        'install': cmd_install,
        'uninstall': cmd_uninstall,
        'status': cmd_status,
    }
    
    handler = commands.get(parsed.command)
    if handler:
        return handler(parsed)
    
    parser.print_help()
    return 1

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| eval: false
#| export __main__
if __name__ == '__main__':
    sys.exit(main())